## basic numpy

arrays, matrices, access elements

In [ ]:
import numpy as np

In [ ]:
l = [1,2,4,10]
a = np.array(l)
a

In [ ]:
a[3] = 4
a

In [ ]:
a.sum()

In [ ]:
a.mean()

In [ ]:
lol = [[1,2,3,4],[5,6,7,8],[9,10,11,12]]
m = np.array(lol)
m

In [ ]:
m.shape

In [ ]:
m.sum()

In [ ]:
m.sum(0)

In [ ]:
m.sum(1)

In [ ]:
m + m

In [ ]:
m.dot(m.T)

In [ ]:
m[1]

In [ ]:
m[:,1]

## installing transformers and loading LLMs

In [ ]:
!pip install transformers
import transformers

In [ ]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained("bert-base-cased")

## tokenization

In [ ]:
text = "Replace me by any text you'd like."
encoded_input = tokenizer(text, return_tensors='pt')

In [ ]:
encoded_input

In [ ]:
encoded_input['input_ids']

In [ ]:
encoded_input['input_ids'].numpy()

In [ ]:
encoded_input['input_ids'].numpy()[0]

In [ ]:
for i in encoded_input['input_ids'].numpy()[0]:
    print(i, tokenizer.decode(i))

## computing contextual vectors

In [ ]:
output = model(**encoded_input)

In [ ]:
encoded_input['input_ids'].shape, output['last_hidden_state'].shape

In [ ]:
M = output['last_hidden_state'].detach().numpy()

In [ ]:
M[:,11,:][0]

## retrieving a specific token

put in np matrix

In [ ]:
sentences = ['The dog can bark loudly.', 
             'He scraped the bark off.',
             'The bark of this tree is not healthy.']
bark_vectors = []
for s in sentences:
    encoded_input = tokenizer(s, return_tensors='pt')

    bark_token = next((i for i,encoding in enumerate(encoded_input['input_ids'][0])
                       if tokenizer.decode(encoding).startswith('bark')),None)
    print(bark_token)
    output = model(**encoded_input)
    vectors = output['last_hidden_state'].detach().numpy()
    bark_vector = vectors[:,bark_token,:][0]
    bark_vectors.append(bark_vector)

In [ ]:
bark_vectors = np.array(bark_vectors)

## cosine similarity

nearest neighbours

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(bark_vectors)

## lexical substitution

## another model: GPT2

In [ ]:
from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

## surprisal in GPT2

In [ ]:
# from: https://github.com/tmalsburg/llm_surprisal/blob/main/llm_generate.py

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn.functional as F

def surprisal(input_text):
  input_tokens = tokenizer.encode(input_text, return_tensors='pt').to(device)
  with torch.no_grad():
    outputs = model(input_tokens, labels=input_tokens)
    logits = outputs.logits

  # Compute log probabilities for all tokens:
  log_probs = F.log_softmax(logits, dim=-1)

  # Shift input_tokens to get surprisal for all tokens, including
  # the first one:
  shifted_tokens = input_tokens[..., 1:]

  # Gather the log probabilities of target tokens:
  target_log_probs = log_probs[:, :-1].gather(2, shifted_tokens.unsqueeze(-1)).squeeze(-1)

  # Compute surprisal (-log probability):
  surprisals = -target_log_probs / torch.log(torch.tensor(2.0))

  # Handle the first token separately:
  first_token = input_tokens[0, 0].item()
  first_token_surprisal = (-log_probs[0, 0, first_token].item() / torch.log(torch.tensor(2.0))).item()

  # Convert token IDs to readable tokens:
  # Note: tolist does not return a list when there's just one token,
  # but a plain int.
  it = input_tokens.squeeze().tolist()
  if type(it) == int:
    it = [it]
  decoded_tokens = [tokenizer.decode([tok]) for tok in it]

  # Include the first token's surprisal:
  surprisals = [first_token_surprisal] + surprisals.squeeze(0).tolist()

  return list(zip(decoded_tokens, surprisals))

In [ ]:
surprisal("I read it in an ancient text.")

In [ ]:
surprisal("I stirred the soup with my text.")